In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
import numpy as np
import re

# -----------------------------
# 1. Load dirty dataset
# -----------------------------
df = pd.read_csv("/content/drive/MyDrive/DWDM_Project/data/raw/samsung_dirty_dataset.csv")

print("Original Shape:", df.shape)

# -----------------------------
# 2. Remove duplicates
# -----------------------------
df = df.drop_duplicates()
df = df.drop_duplicates(subset=['Product', 'Review'])

# -----------------------------
# 3. Remove missing values
# -----------------------------
df = df.dropna(subset=['Product', 'Review', 'Rating'])

# -----------------------------
# 4. Fix rating format
# -----------------------------
df['Rating'] = df['Rating'].astype(str)
df['Rating'] = df['Rating'].str.extract('(\d+\.?\d*)')[0]
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

# -----------------------------
# 5. Remove invalid ratings
# -----------------------------
df = df[(df['Rating'] >= 1) & (df['Rating'] <= 5)]

# -----------------------------
# 6. Clean review text
# -----------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # improved
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Review'] = df['Review'].apply(clean_text)

# -----------------------------
# 7. Remove empty reviews
# -----------------------------
df = df[df['Review'].str.strip() != ""]

# -----------------------------
# 8. Standardize category (FIXED)
# -----------------------------
def assign_category(product):
    name = str(product).lower()

    if 'galaxy' in name or 'phone' in name:
        return 'Smartphone'
    elif 'tab' in name:
        return 'Tablet'
    elif 'buds' in name or 'ear' in name:
        return 'Accessories'
    elif 'charger' in name or 'adapter' in name:
        return 'Accessories'
    elif 'tv' in name:
        return 'TV'
    elif 'book' in name or 'laptop' in name:
        return 'Laptop'
    else:
        return 'Other'

df['Category'] = df['Product'].apply(assign_category)

# -----------------------------
# 9. Add missing columns
# -----------------------------
df['Platform'] = np.random.choice(['Amazon', 'Flipkart'], len(df))
df['Date'] = pd.to_datetime("2023-01-01")

# -----------------------------
# 10. Final structure
# -----------------------------
df = df[['Product', 'Category', 'Rating', 'Review', 'Platform', 'Date']]

# -----------------------------
# 11. Save FINAL RAW dataset
# -----------------------------
df.to_csv("/content/drive/MyDrive/DWDM_Project/data/raw/samsung_clean_dataset.csv", index=False)

print("Final Shape:", df.shape)

<>:27: SyntaxWarning: invalid escape sequence '\d'
<>:27: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_1843/846351660.py:27: SyntaxWarning: invalid escape sequence '\d'
  df['Rating'] = df['Rating'].str.extract('(\d+\.?\d*)')[0]


Original Shape: (11000, 6)
Final Shape: (7494, 6)


In [5]:
print(df.shape)

(7494, 6)
